*0.1 Python for GenAI*

# dataclasses

**The situation.** Your service tracks cost. For every one of 50,000 requests an hour it builds a small record: prompt tokens, completion tokens, model. A developer made that record a Pydantic model "to be safe". Profiling shows 8% of CPU spent validating two integers that came from your own code and were never in doubt.

**The fix: a plain record for data you already trust.** A *dataclass* is a class that just holds fields. Python writes the boring parts (creating, comparing, printing) for you, and nothing is checked — because nothing needs to be. Pydantic at the doors; dataclasses inside.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The price table and the usage record.** `frozen=True` on the price table means it cannot be changed by accident.

In [2]:
from dataclasses import asdict, dataclass, field


@dataclass(frozen=True)
class ModelPrice:
    model: str
    usd_per_million_input: float
    usd_per_million_output: float


@dataclass
class Usage:
    prompt_tokens: int
    completion_tokens: int

    def cost_usd(self, price: ModelPrice) -> float:
        return (
            self.prompt_tokens * price.usd_per_million_input
            + self.completion_tokens * price.usd_per_million_output
        ) / 1e6


@dataclass
class RequestRecord:
    request_id: str
    usage: Usage
    tags: list[str] = field(default_factory=list)  # a fresh list per record


price = ModelPrice("gpt-4o-mini", 0.15, 0.60)
record = RequestRecord("req-8841", Usage(prompt_tokens=1200, completion_tokens=300), tags=["chat"])
print(record)
print("cost: $", round(record.usage.cost_usd(price), 6))
print("as a dictionary for logging:", asdict(record))
assert abs(record.usage.cost_usd(price) - 0.00036) < 1e-9

RequestRecord(request_id='req-8841', usage=Usage(prompt_tokens=1200, completion_tokens=300), tags=['chat'])
cost: $ 0.00036
as a dictionary for logging: {'request_id': 'req-8841', 'usage': {'prompt_tokens': 1200, 'completion_tokens': 300}, 'tags': ['chat']}


**Reading the output.** Printing a record shows every field by name; `asdict` turns it into a plain dictionary for a log line. The cost came out at $0.00036 for 1,200 + 300 tokens. No validation ran — none was needed.

**Frozen means frozen.** Try to change the price table:

In [3]:
try:
    price.usd_per_million_input = 99.0
except Exception as error:
    print("changing a frozen record →", type(error).__name__)
print("two records with the same values are equal:", ModelPrice("gpt-4o-mini", 0.15, 0.60) == price)
assert ModelPrice("gpt-4o-mini", 0.15, 0.60) == price

changing a frozen record → FrozenInstanceError
two records with the same values are equal: True


**The rule to remember.** Data from outside → Pydantic (checked). Data you made yourself, passed around inside → dataclass (fast, plain).

| Use it when | Don't when | Instead use |
|---|---|---|
| internal records: prices, usage, results, config already validated | data from users, files or models — nothing is checked | Pydantic models at every boundary |

**Watch out**
- `tags: list = []` would share one list between every record. Always `field(default_factory=list)`.
- `frozen=True` for anything that must not change (prices, settings) — it also lets the record be used as a dictionary key.
- If you catch yourself adding checks inside a dataclass, the data was not trusted. Make it a Pydantic model.